# NB 01: Exploratory Analysis

Descriptive statistics, distributions, autocorrelation, stationarity,
and cross-exchange correlation of funding rates.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.config import ASSETS, EXCHANGES, PROCESSED_DIR, FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

# Load normalized 8h funding rates
df = pd.read_parquet(PROCESSED_DIR / 'funding_rates_8h.parquet')
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
print(f'Loaded {len(df):,} observations')
print(f'Exchanges: {sorted(df["exchange"].unique())}')
print(f'Assets: {sorted(df["asset"].unique())}')
print(f'Date range: {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')

## 1. Descriptive Statistics

In [ ]:
desc = df.groupby(['exchange', 'asset'])['funding_rate_8h'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    skew='skew',
    kurtosis=lambda x: x.kurtosis(),
    min='min',
    max='max',
    pct_positive=lambda x: (x > 0).mean() * 100,
).round(6)

desc

## 2. Time Series Plots

In [ ]:
for asset in sorted(df['asset'].unique()):
    fig, ax = plt.subplots(figsize=(14, 4))
    asset_data = df[df['asset'] == asset]
    
    for exchange in sorted(asset_data['exchange'].unique()):
        ex_data = asset_data[asset_data['exchange'] == exchange]
        ax.plot(ex_data['timestamp'], ex_data['funding_rate_8h'] * 100,
                label=exchange, alpha=0.7, linewidth=0.5)
    
    ax.set_title(f'{asset} Funding Rates (8h) Across Exchanges')
    ax.set_ylabel('Funding Rate (%)')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'funding_rates_ts_{asset}.pdf', bbox_inches='tight')
    plt.show()

## 3. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(len(df['asset'].unique()), 1,
                         figsize=(12, 3 * len(df['asset'].unique())))
if not hasattr(axes, '__iter__'):
    axes = [axes]

for i, asset in enumerate(sorted(df['asset'].unique())):
    ax = axes[i]
    asset_data = df[df['asset'] == asset]
    
    for exchange in sorted(asset_data['exchange'].unique()):
        rates = asset_data[asset_data['exchange'] == exchange]['funding_rate_8h'] * 100
        rates.hist(ax=ax, bins=80, alpha=0.4, label=exchange, density=True)
    
    ax.set_title(f'{asset} Funding Rate Distribution')
    ax.set_xlabel('Funding Rate (%)')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'funding_rate_distributions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Normality tests
normality = []
for (exchange, asset), group in df.groupby(['exchange', 'asset']):
    rates = group['funding_rate_8h'].dropna()
    if len(rates) < 20:
        continue
    
    # Shapiro-Wilk (subsample if too large)
    sample = rates.sample(min(5000, len(rates)), random_state=42)
    sw_stat, sw_p = stats.shapiro(sample)
    
    # Jarque-Bera
    jb_stat, jb_p = stats.jarque_bera(rates)
    
    normality.append({
        'exchange': exchange, 'asset': asset,
        'shapiro_stat': sw_stat, 'shapiro_p': sw_p,
        'jarque_bera_stat': jb_stat, 'jarque_bera_p': jb_p,
        'is_normal_sw': sw_p > 0.05, 'is_normal_jb': jb_p > 0.05,
    })

normality_df = pd.DataFrame(normality)
print('Normality Tests (p > 0.05 = normal):')
print(f'Normal by Shapiro-Wilk: {normality_df["is_normal_sw"].sum()}/{len(normality_df)}')
print(f'Normal by Jarque-Bera: {normality_df["is_normal_jb"].sum()}/{len(normality_df)}')
normality_df

## 4. Autocorrelation (ACF/PACF)

Persistent autocorrelation suggests exploitable patterns.

In [ ]:
# ACF/PACF for BTC on each exchange
btc = df[df['asset'] == 'BTC']

for exchange in sorted(btc['exchange'].unique()):
    rates = btc[btc['exchange'] == exchange].set_index('timestamp')['funding_rate_8h'].dropna()
    if len(rates) < 50:
        continue
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
    plot_acf(rates, ax=ax1, lags=30, title=f'{exchange} BTC ACF')
    plot_pacf(rates, ax=ax2, lags=30, title=f'{exchange} BTC PACF')
    plt.tight_layout()
    plt.show()

## 5. Stationarity Tests (ADF / KPSS)

In [ ]:
stationarity = []
for (exchange, asset), group in df.groupby(['exchange', 'asset']):
    rates = group.set_index('timestamp')['funding_rate_8h'].dropna()
    if len(rates) < 50:
        continue
    
    # ADF test (H0: unit root / non-stationary)
    adf_stat, adf_p, *_ = adfuller(rates, maxlag=20)
    
    # KPSS test (H0: stationary)
    kpss_stat, kpss_p, *_ = kpss(rates, regression='c', nlags='auto')
    
    stationarity.append({
        'exchange': exchange, 'asset': asset,
        'adf_stat': adf_stat, 'adf_p': adf_p,
        'kpss_stat': kpss_stat, 'kpss_p': kpss_p,
        'stationary_adf': adf_p < 0.05,
        'stationary_kpss': kpss_p > 0.05,
        'conclusion': 'stationary' if (adf_p < 0.05 and kpss_p > 0.05) else
                      'non-stationary' if (adf_p >= 0.05 and kpss_p <= 0.05) else
                      'ambiguous'
    })

stat_df = pd.DataFrame(stationarity)
print('Stationarity Results:')
print(stat_df.groupby('conclusion').size())
stat_df

## 6. Cross-Exchange Correlation Matrix

In [ ]:
# Load aligned wide-format data
wide = pd.read_parquet(PROCESSED_DIR / 'funding_rates_aligned.parquet')

for asset in sorted(df['asset'].unique()):
    # Extract columns for this asset
    cols = [(ex, a) for ex, a in wide.columns if a == asset]
    if len(cols) < 2:
        continue
    
    asset_wide = wide[cols].dropna()
    asset_wide.columns = [ex for ex, _ in cols]
    
    if asset_wide.empty:
        continue
    
    corr = asset_wide.corr()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'{asset} Funding Rate Cross-Exchange Correlation')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'correlation_{asset}.pdf', bbox_inches='tight')
    plt.show()

## 7. Regime Analysis

Segment into bull/bear/sideways using BTC price trend.

In [ ]:
from pathlib import Path
from src.config import RAW_DIR

# Try to load BTC price data
price_files = list((RAW_DIR / 'prices').rglob('BTC.parquet'))
if price_files:
    prices = pd.read_parquet(price_files[0])
    prices['timestamp'] = pd.to_datetime(prices['timestamp'], utc=True)
    prices = prices.set_index('timestamp').sort_index()
    
    # 30-day rolling return to classify regime
    prices['ret_30d'] = prices['close'].pct_change(periods=90)  # 90 hourly candles ≈ 30 8h periods
    prices['regime'] = pd.cut(prices['ret_30d'],
                              bins=[-np.inf, -0.05, 0.05, np.inf],
                              labels=['bear', 'sideways', 'bull'])
    
    fig, ax = plt.subplots(figsize=(14, 4))
    colors = {'bull': 'green', 'bear': 'red', 'sideways': 'gray'}
    for regime in ['bull', 'bear', 'sideways']:
        mask = prices['regime'] == regime
        ax.scatter(prices.index[mask], prices['close'][mask],
                  c=colors[regime], s=0.5, label=regime, alpha=0.5)
    ax.set_title('BTC Price with Market Regime')
    ax.set_ylabel('Price (USD)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'btc_regime.pdf', bbox_inches='tight')
    plt.show()
    
    # Funding rate stats by regime
    prices_8h = prices['regime'].resample('8h').last().dropna()
    btc_rates = df[df['asset'] == 'BTC'].set_index('timestamp')
    btc_rates = btc_rates.join(prices_8h, how='left')
    
    if btc_rates['regime'].notna().any():
        regime_stats = btc_rates.groupby(['exchange', 'regime'])['funding_rate_8h'].agg(
            ['mean', 'std', 'count']
        ).round(6)
        print('Funding rates by regime:')
        print(regime_stats)
else:
    print('No price data available for regime analysis. Run NB 00 price collection first.')